##  DateTime Operations

### Why does this matter?

Almost every real dataset has dates — order dates, transaction timestamps, patient admission dates, sensor readings.

 Time is one of the most powerful dimensions in data analysis. "Sales this month vs last month", "which hour has most orders", "how many days since last purchase" — all of these require DateTime operations.
 
  If you can't work with dates, you can't do time-based analysis, which is a huge part of a Data Analyst's job.

### What is a DateTime in Pandas?

What is it?
Pandas has a special data type called *datetime64* for storing dates and times. It is NOT a regular string like '2024-01-15'.

 It is a numeric value internally — representing the number of nanoseconds since January 1, 1970 (called Unix epoch). This is why you can do math on dates — subtract two dates to get days, add 30 days to a date, etc.

### Why convert strings to datetime?

Because raw data almost always has dates stored as strings — '2024-01-15', '15/01/2024', 'Jan 15 2024'. A string is just text — you can't do math on it, you can't extract the month, you can't sort it correctly. Converting to datetime64 unlocks all of that.

### How does *pd.to_datetime()* work?

Pandas reads the string, detects the date format (or you specify it), and converts it to a *datetime64* value internally. After that, the .*dt* accessor gives you access to all date components — just like *.str* gives you string methods.

In [1]:
import pandas as pd

data = {
    'OrderID':   [1, 2, 3, 4, 5],
    'Customer':  ['Sumed', 'Priya', 'Rahul', 'Neha', 'Arjun'],
    'OrderDate': ['2024-01-15', '2024-03-22', '2024-03-05',
                  '2024-07-18', '2024-11-30'],
    'DelivDate': ['2024-01-20', '2024-03-28', '2024-03-12',
                  '2024-07-25', '2024-12-05'],
    'Amount':    [1500, 3200, 800, 4500, 2100]
}
df = pd.DataFrame(data)
print(df.dtypes)
# OrderDate → object (string) — not datetime yet!

OrderID       int64
Customer     object
OrderDate    object
DelivDate    object
Amount        int64
dtype: object


*When you read a CSV, date columns come in as object (string) by default. You MUST convert them to datetime before doing any date operations. This is one of the most common beginner mistakes.*

## PD.TO_DATETIME() : converting string to datetime

In [2]:
# pd.to_datetime() → converts string column to datetime64 type
# Pandas auto-detects common formats like YYYY-MM-DD

df['OrderDate'] = pd.to_datetime(df['OrderDate'])
df['DelivDate'] = pd.to_datetime(df['DelivDate'])

print(df.dtypes)
# Now both are datetime64[ns] — ns = nanoseconds precision

# When format is non-standard, specify it explicitly
# format codes: %d=day, %m=month, %Y=4-digit year, %y=2-digit year
# %H=hour, %M=minute, %S=second

pd.to_datetime('15/01/2024', format='%d/%m/%Y')
pd.to_datetime('Jan 15 2024', format='%b %d %Y')
pd.to_datetime('2024-01-15 14:30:00')  # with time — auto-detected

# errors='coerce' → converts bad/unreadable dates to NaT (Not a Time)
# instead of crashing on bad values
pd.to_datetime(df['OrderDate'], errors='coerce')

OrderID               int64
Customer             object
OrderDate    datetime64[ns]
DelivDate    datetime64[ns]
Amount                int64
dtype: object


0   2024-01-15
1   2024-03-22
2   2024-03-05
3   2024-07-18
4   2024-11-30
Name: OrderDate, dtype: datetime64[ns]

*Always use errors='coerce' in production code — real datasets have corrupted date values. NaT (Not a Time) is the datetime equivalent of NaN — you can handle it the same way with isnull().*


**pd.to_datetime('15/01/2024', format='%d/%m/%Y'):**

**Timestamp('2024-01-15 00:00:00')**

## .DT ACCESSOR : extracting date components

In [4]:
# .dt accessor → like .str for strings, but for datetime columns
# Gives access to all date/time components

df['OrderDate'].dt.year         # → 2024 for all rows
df['OrderDate'].dt.month        # → 1, 3, 3, 7, 11
df['OrderDate'].dt.day          # → 15, 22, 5, 18, 30
df['OrderDate'].dt.dayofweek    # → 0=Monday ... 6=Sunday
df['OrderDate'].dt.day_name()   # → 'Monday', 'Friday' etc.
df['OrderDate'].dt.month_name() # → 'January', 'March' etc.
df['OrderDate'].dt.quarter      # → 1, 1, 1, 3, 4
#df['OrderDate'].dt.weekofyear   # → week number in year
df['OrderDate'].dt.is_month_end  # → True if last day of month

# Add useful columns to the DataFrame
df['Order_Month']   = df['OrderDate'].dt.month
df['Order_Quarter'] = df['OrderDate'].dt.quarter
df['Order_DayName'] = df['OrderDate'].dt.day_name()

In [5]:
df

,OrderID,Customer,OrderDate,DelivDate,Amount,Order_Month,Order_Quarter,Order_DayName
0,1,Sumed,2024-01-15,2024-01-20,1500,1,1,Monday
1,2,Priya,2024-03-22,2024-03-28,3200,3,1,Friday
2,3,Rahul,2024-03-05,2024-03-12,800,3,1,Tuesday
3,4,Neha,2024-07-18,2024-07-25,4500,7,3,Thursday
4,5,Arjun,2024-11-30,2024-12-05,2100,11,4,Saturday


 *Extracting month, quarter, day_name then grouping by them is one of the most common patterns in business reporting — "sales by month", "orders by weekday", "revenue by quarter".*

## DATE ARITHMETIC : timedelta

In [8]:
# Subtracting two datetime columns gives a Timedelta
# Timedelta = duration between two timestamps
# This is why datetime is stored as numbers internally — math works

df['DeliveryDays'] = df['DelivDate'] - df['OrderDate']
print(df['DeliveryDays'])
# Output: 5 days, 6 days, 7 days... (Timedelta objects)



0   5 days
1   6 days
2   7 days
3   7 days
4   5 days
Name: DeliveryDays, dtype: timedelta64[ns]


In [10]:
# Extract just the NUMBER of days from Timedelta
df['DeliveryDays'] = (df['DelivDate'] - df['OrderDate']).dt.days
# .dt.days extracts integer day count from Timedelta


In [13]:
df['DeliveryDays']

0    5
1    6
2    7
3    7
4    5
Name: DeliveryDays, dtype: int64

In [11]:

# Days since today (age of order)
today = pd.Timestamp('today')
df['DaysSinceOrder'] = (today - df['OrderDate']).dt.days



In [16]:
df['DaysSinceOrder']

0    795
1    728
2    745
3    610
4    475
Name: DaysSinceOrder, dtype: int64

In [12]:
# Adding days to a date → use pd.Timedelta or DateOffset
df['ExpectedDelivery'] = df['OrderDate'] + pd.Timedelta(days=7)
df['NextMonth']       = df['OrderDate'] + pd.DateOffset(months=1)

In [15]:
df['ExpectedDelivery']

0   2024-01-22
1   2024-03-29
2   2024-03-12
3   2024-07-25
4   2024-12-07
Name: ExpectedDelivery, dtype: datetime64[ns]

In [17]:
df['NextMonth']

0   2024-02-15
1   2024-04-22
2   2024-04-05
3   2024-08-18
4   2024-12-30
Name: NextMonth, dtype: datetime64[ns]

*Timedelta = result of subtracting two datetimes. .dt.days extracts the integer. pd.Timedelta(days=7) adds a fixed duration. pd.DateOffset(months=1) adds calendar months (handles month-length differences correctly).*

## Filtering by date

In [22]:
# After converting to datetime, you can filter using comparison operators
# Strings in comparisons are auto-converted to datetime by Pandas

# Orders after March 1 2024
df[df['OrderDate'] > '2024-03-01']




,OrderID,Customer,OrderDate,DelivDate,Amount,Order_Month,Order_Quarter,Order_DayName,DeliveryDays,DaysSinceOrder,ExpectedDelivery,NextMonth
1,2,Priya,2024-03-22,2024-03-28,3200,3,1,Friday,6,728,2024-03-29,2024-04-22
2,3,Rahul,2024-03-05,2024-03-12,800,3,1,Tuesday,7,745,2024-03-12,2024-04-05
3,4,Neha,2024-07-18,2024-07-25,4500,7,3,Thursday,7,610,2024-07-25,2024-08-18
4,5,Arjun,2024-11-30,2024-12-05,2100,11,4,Saturday,5,475,2024-12-07,2024-12-30


In [23]:
# Orders in Q1 (Jan-Mar)
df[df['OrderDate'].dt.quarter == 1]

,OrderID,Customer,OrderDate,DelivDate,Amount,Order_Month,Order_Quarter,Order_DayName,DeliveryDays,DaysSinceOrder,ExpectedDelivery,NextMonth
0,1,Sumed,2024-01-15,2024-01-20,1500,1,1,Monday,5,795,2024-01-22,2024-02-15
1,2,Priya,2024-03-22,2024-03-28,3200,3,1,Friday,6,728,2024-03-29,2024-04-22
2,3,Rahul,2024-03-05,2024-03-12,800,3,1,Tuesday,7,745,2024-03-12,2024-04-05


In [24]:
# Orders in a specific month
df[df['OrderDate'].dt.month == 3]   # March only

,OrderID,Customer,OrderDate,DelivDate,Amount,Order_Month,Order_Quarter,Order_DayName,DeliveryDays,DaysSinceOrder,ExpectedDelivery,NextMonth
1,2,Priya,2024-03-22,2024-03-28,3200,3,1,Friday,6,728,2024-03-29,2024-04-22
2,3,Rahul,2024-03-05,2024-03-12,800,3,1,Tuesday,7,745,2024-03-12,2024-04-05


In [25]:
# Orders between two dates
df[df['OrderDate'].between('2024-01-01', '2024-06-30')]

,OrderID,Customer,OrderDate,DelivDate,Amount,Order_Month,Order_Quarter,Order_DayName,DeliveryDays,DaysSinceOrder,ExpectedDelivery,NextMonth
0,1,Sumed,2024-01-15,2024-01-20,1500,1,1,Monday,5,795,2024-01-22,2024-02-15
1,2,Priya,2024-03-22,2024-03-28,3200,3,1,Friday,6,728,2024-03-29,2024-04-22
2,3,Rahul,2024-03-05,2024-03-12,800,3,1,Tuesday,7,745,2024-03-12,2024-04-05


In [26]:
# Orders on weekends
df[df['OrderDate'].dt.dayofweek >= 5]  # 5=Saturday, 6=Sunday

,OrderID,Customer,OrderDate,DelivDate,Amount,Order_Month,Order_Quarter,Order_DayName,DeliveryDays,DaysSinceOrder,ExpectedDelivery,NextMonth
4,5,Arjun,2024-11-30,2024-12-05,2100,11,4,Saturday,5,475,2024-12-07,2024-12-30


## Groupby with datetime components

In [30]:
# Most powerful pattern: extract component → groupby → aggregate
# This is how all time-based business reports are built

# Total sales by month
df.groupby(df['OrderDate'].dt.month)['Amount'].sum()


OrderDate
1     1500
3     4000
7     4500
11    2100
Name: Amount, dtype: int64

In [31]:

# Total sales by quarter
df.groupby(df['OrderDate'].dt.quarter)['Amount'].sum()



OrderDate
1    5500
3    4500
4    2100
Name: Amount, dtype: int64

In [32]:
# Average delivery days by month
df.groupby(df['OrderDate'].dt.month)['DeliveryDays'].mean()



OrderDate
1     5.0
3     6.5
7     7.0
11    5.0
Name: DeliveryDays, dtype: float64

In [33]:
# Order count by day of week — which day gets most orders?
df.groupby(df['OrderDate'].dt.day_name())['OrderID'].count()

OrderDate
Friday      1
Monday      1
Saturday    1
Thursday    1
Tuesday     1
Name: OrderID, dtype: int64

*This exact pattern — groupby(df['date'].dt.month)['value'].sum() — is used in almost every business dashboard. Monthly revenue, quarterly growth, weekly active users — all built this way.*

## RESAMPLE() : time based aggregation

In [36]:
# resample() → like groupby but specifically for time series
# Requires datetime column to be the INDEX
# Aggregates by time frequency

df2 = df.set_index('OrderDate')  # set date as index first



In [37]:
# Resample by Month — 'ME' = Month End
df2['Amount'].resample('ME').sum()



OrderDate
2024-01-31    1500
2024-02-29       0
2024-03-31    4000
2024-04-30       0
2024-05-31       0
2024-06-30       0
2024-07-31    4500
2024-08-31       0
2024-09-30       0
2024-10-31       0
2024-11-30    2100
Freq: ME, Name: Amount, dtype: int64

In [38]:
# Resample by Quarter — 'QE' = Quarter End
df2['Amount'].resample('QE').sum()

# Common frequency strings:
# 'D'  → daily
# 'W'  → weekly
# 'ME' → monthly (month end)
# 'QE' → quarterly (quarter end)
# 'YE' → yearly (year end)
# 'H'  → hourly (for timestamp data)

# WHEN to use resample vs groupby + dt.month?
# resample → when df already has datetime index, filling gaps with 0
# groupby  → when date is a regular column, more flexible

OrderDate
2024-03-31    5500
2024-06-30       0
2024-09-30    4500
2024-12-31    2100
Freq: QE-DEC, Name: Amount, dtype: int64

In [39]:
df2

,OrderID,Customer,DelivDate,Amount,Order_Month,Order_Quarter,Order_DayName,DeliveryDays,DaysSinceOrder,ExpectedDelivery,NextMonth
OrderDate,,,,,,,,,,,
2024-01-15,1,Sumed,2024-01-20,1500,1,1,Monday,5,795,2024-01-22,2024-02-15
2024-03-22,2,Priya,2024-03-28,3200,3,1,Friday,6,728,2024-03-29,2024-04-22
2024-03-05,3,Rahul,2024-03-12,800,3,1,Tuesday,7,745,2024-03-12,2024-04-05
2024-07-18,4,Neha,2024-07-25,4500,7,3,Thursday,7,610,2024-07-25,2024-08-18
2024-11-30,5,Arjun,2024-12-05,2100,11,4,Saturday,5,475,2024-12-07,2024-12-30


*resample() fills in missing months with 0 automatically — Feb shows 0 because there were no orders. groupby + dt.month would skip Feb entirely. This matters for charts and trend analysis.*

## Real world pipeline

In [40]:
# Complete datetime cleaning + analysis workflow

# Step 1: Convert string columns to datetime
df['OrderDate'] = pd.to_datetime(df['OrderDate'], errors='coerce')
df['DelivDate'] = pd.to_datetime(df['DelivDate'], errors='coerce')

# Step 2: Check for NaT (failed conversions)
df['OrderDate'].isnull().sum()

# Step 3: Extract useful components
df['Year']     = df['OrderDate'].dt.year
df['Month']    = df['OrderDate'].dt.month
df['Quarter']  = df['OrderDate'].dt.quarter
df['DayName']  = df['OrderDate'].dt.day_name()

# Step 4: Calculate derived metrics
df['DelivDays'] = (df['DelivDate'] - df['OrderDate']).dt.days

# Step 5: Aggregate for report
monthly = df.groupby('Month').agg(
    total_sales  = ('Amount',    'sum'),
    order_count  = ('OrderID',   'count'),
    avg_delivery = ('DelivDays', 'mean')
).reset_index()
print(monthly)

   Month  total_sales  order_count  avg_delivery
0      1         1500            1           5.0
1      3         4000            2           6.5
2      7         4500            1           7.0
3     11         2100            1           5.0


*This exact workflow — convert → extract → aggregate — is what you'll do in every real project that has a date column. It directly maps to what you'd present in a Power BI dashboard.*

## Everything You Must Know Cold


**Dates come in as strings** — when you read a CSV, date columns are object type by *default*. Always convert with *pd.to_datetime()* before any date operation.

**pd.to_datetime(errors='coerce')** — always use *errors='coerce'* in production. Bad date values become *NaT* instead of crashing your code.

**NaT** — Not a Time. The datetime equivalent of *NaN*. Handled identically with *isnull()* and *fillna()*.

**.dt accessor** — like .str for strings, .dt unlocks all datetime components. Only works after converting to datetime64.

**Timedelta** — result of subtracting two datetime columns. Use *.dt.days* to extract the integer day count.

**pd.Timedelta(days=N)** — adds fixed *days. pd.DateOffset(months=N)* — adds calendar months (handles 28/30/31 day months correctly).

**resample()** — like groupby but time-aware. Fills missing time periods with 0. Requires datetime index. Use **'ME', 'QE', 'YE'** for month/quarter/year.

**groupby(df['date'].dt.month)** — most common time analysis pattern. Extract component then group.